In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json

# ==========================================
# Event Hub Connection
# ==========================================

connectionString = (
    "Endpoint=sb://eh-social-media.servicebus.windows.net/;"
    "SharedAccessKeyName=sudheer;"
    "SharedAccessKey=1IhrynvVGzNbhQLyvqMUxuT7QdTRHRh57+AEhJK7+eQ=;"
    "EntityPath=sentiment-hub"
)

ehConf = {}

ehConf["eventhubs.connectionString"] = (
    sc._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(connectionString)
)

ehConf["eventhubs.consumerGroup"] = "$Default"

startingEventPosition = {
    "offset": "-1",
    "seqNo": -1,
    "enqueuedTime": None,
    "isInclusive": True
}

ehConf["eventhubs.startingPosition"] = json.dumps(startingEventPosition)

# Read up to 5000 events
ehConf["eventhubs.maxEventsPerTrigger"] = "5000"

# ==========================================
# Read Stream
# ==========================================

rawDF = (
    spark.readStream
         .format("eventhubs")
         .options(**ehConf)
         .load()
)

# Convert binary body to string
eventDF = rawDF.selectExpr(
    "CAST(body AS STRING) AS message"
)

# ==========================================
# Schema
# ==========================================

schema = StructType([
    StructField("tweet_id", StringType(), True),
    StructField("topic_category", StringType(), True),
    StructField("tweet_timestamp", StringType(), True),
    StructField("sentiment_score", DoubleType(), True),
    StructField("positive_score", DoubleType(), True),
    StructField("negative_score", DoubleType(), True),
    StructField("neutral_score", DoubleType(), True),
    StructField("impressions", DoubleType(), True),
    StructField("likes", DoubleType(), True),
    StructField("engagement_count", DoubleType(), True)
])

# ==========================================
# Parse JSON
# ==========================================

parsedDF = (
    eventDF
        .select(from_json(col("message"), schema).alias("data"))
        .select("data.*")
)

# ==========================================
# Error Records
# ==========================================

errorDF = parsedDF.filter(col("tweet_id").isNull())

errorQuery = (
    errorDF.writeStream
        .trigger(availableNow=True)
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "abfss://raw@socialmedia1.dfs.core.windows.net/checkpoints/error_bronze_sentiment"
        )
        .toTable("socialmedia_databricks.error.error_bronze_sentiment")
)

# ==========================================
# Bronze Data
# ==========================================

bronzeDF = (
    parsedDF
        .filter(col("tweet_id").isNotNull())
        .withColumn(
            "tweet_timestamp",
            to_timestamp(col("tweet_timestamp"), "dd-MM-yyyy HH:mm")
        )
        .withColumn("bronze_load_time", current_timestamp())
        .withColumn("pipeline_name", lit("Bronze_Sentiment"))
        .withColumn("source_system", lit("Azure Event Hub"))
        .withColumn("ingestion_date", current_date())
        .withWatermark("tweet_timestamp", "10 minutes")
)

# ==========================================
# Write Bronze Table
# ==========================================

bronzeQuery = (
    bronzeDF.writeStream
        .trigger(availableNow=True)
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "abfss://raw@socialmedia1.dfs.core.windows.net/checkpoints/bronze_sentiment1"
        )
        .option("mergeSchema", "true")
        .toTable("socialmedia_databricks.bronze.bronze_sentiment1")
)

bronzeQuery.awaitTermination()
errorQuery.awaitTermination()

print("Bronze Sentiment Loaded Successfully")

Bronze Sentiment Loaded Successfully


In [0]:
%sql SHOW TABLES IN socialmedia_databricks.error;

database,tableName,isTemporary


In [0]:
%sql
SELECT *
FROM socialmedia_databricks.bronze.bronze_sentiment1;

tweet_id,topic_category,tweet_timestamp,sentiment_score,positive_score,negative_score,neutral_score,impressions,likes,engagement_count,bronze_load_time,pipeline_name,source_system,ingestion_date
T36841,Cloud,2025-01-07T19:01:00Z,0.912376086,0.381988998,0.934702541,0.125657227,6126.0,24.0,NaN,2026-07-09T05:08:40.208Z,Bronze_Sentiment,Azure Event Hub,2026-07-09
T27424,Cloud,2025-01-08T19:02:00Z,0.904795167,0.583384394,0.159876995,0.299313398,8630.0,4873.0,541.0,2026-07-09T05:08:40.208Z,Bronze_Sentiment,Azure Event Hub,2026-07-09
T31365,Cloud,2025-01-04T11:51:00Z,0.976751897,0.057764811,0.660596526,0.643261617,15985.0,940.0,973.0,2026-07-09T05:08:40.208Z,Bronze_Sentiment,Azure Event Hub,2026-07-09
T33361,Sports,2025-01-02T11:00:00Z,0.165869516,0.812630732,0.954412139,0.789351236,NaN,4815.0,241.0,2026-07-09T05:08:40.208Z,Bronze_Sentiment,Azure Event Hub,2026-07-09
T35147,Sports,2025-01-21T17:06:00Z,-0.038402341,0.845799773,0.91528097,0.685045982,1760.0,4327.0,678.0,2026-07-09T05:08:40.208Z,Bronze_Sentiment,Azure Event Hub,2026-07-09
T35107,AI,2025-01-09T15:50:00Z,-0.440088528,0.038703165,0.120721452,0.969653285,3340.0,4690.0,657.0,2026-07-09T05:08:40.208Z,Bronze_Sentiment,Azure Event Hub,2026-07-09
T39614,Sports,null,0.519327259,0.140576522,0.470191614,NaN,123.0,3396.0,959.0,2026-07-09T05:08:40.208Z,Bronze_Sentiment,Azure Event Hub,2026-07-09
T25529,AI,2025-01-12T17:23:00Z,0.204135828,0.231063306,0.346975728,0.35930797,17237.0,NaN,78.0,2026-07-09T05:08:40.208Z,Bronze_Sentiment,Azure Event Hub,2026-07-09
T9125,Cloud,2025-01-04T06:11:00Z,0.733573789,0.765106483,0.761823891,0.027961238,11856.0,3466.0,146.0,2026-07-09T05:08:40.208Z,Bronze_Sentiment,Azure Event Hub,2026-07-09
T30613,Finance,2025-01-23T09:04:00Z,0.357681081,0.400286448,0.432258961,0.933616025,3929.0,2886.0,961.0,2026-07-09T05:08:40.208Z,Bronze_Sentiment,Azure Event Hub,2026-07-09
